# Retail Customer PD Scorecard

This notebook demonstrates building a credit scoring model using Random Forest for retail customers.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

print('Libraries imported successfully!')

## 1. Generate Synthetic Retail Customer Data

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Generate synthetic data for 1000 retail customers
n_customers = 1000

# Customer demographics
data = {
    'customer_id': range(1, n_customers + 1),
    'age': np.random.normal(40, 12, n_customers).astype(int),
    'income': np.random.lognormal(10.5, 0.5, n_customers).astype(int),
    'credit_score': np.random.normal(650, 100, n_customers).astype(int),
    'employment_length': np.random.exponential(5, n_customers).astype(int),
    'debt_to_income': np.random.beta(2, 5, n_customers),
    'num_credit_cards': np.random.poisson(2, n_customers),
    'has_mortgage': np.random.choice([0, 1], n_customers, p=[0.4, 0.6]),
    'num_late_payments': np.random.poisson(1, n_customers),
    'credit_utilization': np.random.beta(3, 2, n_customers)
}

# Create DataFrame
df = pd.DataFrame(data)

# Cap values at realistic ranges
df['age'] = df['age'].clip(18, 80)
df['credit_score'] = df['credit_score'].clip(300, 850)
df['debt_to_income'] = df['debt_to_income'].clip(0, 1)
df['credit_utilization'] = df['credit_utilization'].clip(0, 1)

print(f"Generated {len(df)} customer records")
print("\nFirst 5 records:")
df.head()

## 2. Create Target Variable (Default Probability)

In [ ]:
# Calculate default probability based on risk factors
def calculate_default_probability(row):
    # Base probability
    prob = 0.05  # 5% base default rate
    
    # Credit score impact (lower score = higher risk)
    if row['credit_score'] < 500:
        prob += 0.25
    elif row['credit_score'] < 600:
        prob += 0.15
    elif row['credit_score'] < 700:
        prob += 0.05
    
    # Debt to income ratio impact
    if row['debt_to_income'] > 0.5:
        prob += 0.15
    elif row['debt_to_income'] > 0.3:
        prob += 0.05
    
    # Late payments impact
    if row['num_late_payments'] > 3:
        prob += 0.20
    elif row['num_late_payments'] > 1:
        prob += 0.10
    
    # Credit utilization impact
    if row['credit_utilization'] > 0.8:
        prob += 0.15
    elif row['credit_utilization'] > 0.5:
        prob += 0.05
    
    # Income factor (lower income = higher risk)
    if row['income'] < 30000:
        prob += 0.10
    
    return min(prob, 0.95)  # Cap at 95%

# Apply probability calculation
df['default_probability'] = df.apply(calculate_default_probability, axis=1)

# Generate binary default outcome based on probability
df['defaulted'] = np.random.random(len(df)) < df['default_probability']
df['defaulted'] = df['defaulted'].astype(int)

print(f"Default rate: {df['defaulted'].mean():.2%}")
print("\nDefault distribution:")
print(df['defaulted'].value_counts())

## 3. Data Exploration

In [ ]:
# Summary statistics
print("Data Summary:")
df.describe()

In [ ]:
# Visualize key features by default status
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Customer Features by Default Status', fontsize=16)

# Credit Score Distribution
sns.histplot(data=df, x='credit_score', hue='defaulted', ax=axes[0,0], alpha=0.7)
axes[0,0].set_title('Credit Score Distribution')

# Debt to Income Ratio
sns.boxplot(data=df, x='defaulted', y='debt_to_income', ax=axes[0,1])
axes[0,1].set_title('Debt to Income Ratio')

# Credit Utilization
sns.boxplot(data=df, x='defaulted', y='credit_utilization', ax=axes[0,2])
axes[0,2].set_title('Credit Utilization')

# Income Distribution
sns.histplot(data=df, x='income', hue='defaulted', ax=axes[1,0], alpha=0.7)
axes[1,0].set_title('Income Distribution')

# Number of Late Payments
sns.countplot(data=df, x='num_late_payments', hue='defaulted', ax=axes[1,1])
axes[1,1].set_title('Late Payments Count')

# Age Distribution
sns.boxplot(data=df, x='defaulted', y='age', ax=axes[1,2])
axes[1,2].set_title('Age Distribution')

plt.tight_layout()
plt.show()

## 4. Model Preparation

In [ ]:
# Select features for modeling
feature_columns = [
    'age', 'income', 'credit_score', 'employment_length',
    'debt_to_income', 'num_credit_cards', 'has_mortgage',
    'num_late_payments', 'credit_utilization'
]

X = df[feature_columns]
y = df['defaulted']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Training default rate: {y_train.mean():.2%}")
print(f"Test default rate: {y_test.mean():.2%}")

## 5. Random Forest Model Training

In [ ]:
# Initialize and train Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    class_weight='balanced'
)

# Train the model
rf_model.fit(X_train, y_train)

print("Random Forest model trained successfully!")
print(f"Model accuracy on training set: {rf_model.score(X_train, y_train):.3f}")
print(f"Model accuracy on test set: {rf_model.score(X_test, y_test):.3f}")

## 6. Model Evaluation

In [ ]:
# Make predictions
y_pred = rf_model.predict(X_test)
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]

# Print classification report
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Calculate AUC-ROC score
auc_score = roc_auc_score(y_test, y_pred_proba)
print(f"AUC-ROC Score: {auc_score:.3f}")

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No Default', 'Default'],
            yticklabels=['No Default', 'Default'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

## 7. Feature Importance

In [ ]:
# Get feature importance
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

# Plot feature importance
plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance, x='importance', y='feature')
plt.title('Feature Importance in Random Forest Model')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.show()

print("Feature Importance:")
print(feature_importance)

## 8. Create Scorecard Function

In [ ]:
def calculate_score_score(customer_data):
    """
    Calculate credit score for a customer using the trained Random Forest model.
    Returns a score from 300-850 (higher = better creditworthiness)
    """
    # Get default probability
    default_prob = rf_model.predict_proba(customer_data)[0, 1]
    
    # Convert to credit score (inverse relationship)
    # Higher probability = lower score
    credit_score = int(850 - (default_prob * 550))
    credit_score = max(300, min(850, credit_score))  # Clamp to 300-850 range
    
    return credit_score, default_prob

# Test with a few sample customers
sample_customers = X_test.head(3)
print("Sample Customer Scorecards:")
print("=" * 50)

for idx, (i, customer) in enumerate(sample_customers.iterrows()):
    credit_score, default_prob = calculate_score_score(customer.values.reshape(1, -1))
    actual_default = y_test.iloc[i]
    
    print(f"\nCustomer {idx+1}:")
    print(f"  Credit Score: {credit_score}")
    print(f"  Default Probability: {default_prob:.2%}")
    print(f"  Actual Default: {'Yes' if actual_default else 'No'}")
    print(f"  Key Features: Credit Score={customer['credit_score']:.0f}, DTI={customer['debt_to_income']:.2f}")

## 9. Save Model and Data

In [ ]:
import joblib

# Save the trained model
joblib.dump(rf_model, 'random_forest_scorecard_model.pkl')

# Save the dataset
df.to_csv('retail_customer_data.csv', index=False)

# Save feature list
with open('feature_list.txt', 'w') as f:
    f.write('\n'.join(feature_columns))

print("Model and data saved successfully!")
print("- random_forest_scorecard_model.pkl")
print("- retail_customer_data.csv")
print("- feature_list.txt")

## 10. Summary

This notebook demonstrates:
1. **Data Generation**: Created synthetic retail customer data with relevant credit features
2. **Feature Engineering**: Built meaningful risk factors for default prediction
3. **Model Training**: Used Random Forest for binary classification
4. **Evaluation**: Assessed model performance with accuracy, AUC-ROC, and confusion matrix
5. **Scorecard Creation**: Built a practical credit scoring function
6. **Interpretability**: Analyzed feature importance for model transparency

### Key Results:
- Model accuracy and AUC-ROC scores
- Most important risk factors identified
- Working credit score calculation function
- Saved model for future use